# Tokenization Comparison – Airline Tweets
Regex vs NLTK vs spaCy

In [1]:
from tokenization import *
import pandas as pd
pd.set_option('display.max_colwidth', None)
df = load_tweets()
texts = df['text'].head(8).tolist()
len(df)

/Users/minahil/NLP-Project/NLP-Project/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loaded: data/raw/Tweets.csv | rows: 14640


14640

## 1. Test on actual tweets

In [2]:
compare(texts[:5])[['tweet','Regex','NLTK','spaCy']]

,tweet,Regex,NLTK,spaCy
0,@VirginAmerica What @dhepburn said.,"[@VirginAmerica, What, @dhepburn, said, .]","[@, VirginAmerica, What, @, dhepburn, said, .]","[@VirginAmerica, What, @dhepburn, said, .]"
1,@VirginAmerica plus you've added commercials to the experience... tacky.,"[@VirginAmerica, plus, you've, added, commercials, to, the, experience, ., ., ., tacky, .]","[@, VirginAmerica, plus, you, 've, added, commercials, to, the, experience, ..., tacky, .]","[@VirginAmerica, plus, you, 've, added, commercials, to, the, experience, ..., tacky, .]"
2,@VirginAmerica I didn't today... Must mean I need to take another trip!,"[@VirginAmerica, I, didn't, today, ., ., ., Must, mean, I, need, to, take, another, trip, !]","[@, VirginAmerica, I, did, n't, today, ..., Must, mean, I, need, to, take, another, trip, !]","[@VirginAmerica, I, did, n't, today, ..., Must, mean, I, need, to, take, another, trip, !]"
3,"@VirginAmerica it's really aggressive to blast obnoxious ""entertainment"" in your guests' faces &amp; they have little recourse","[@VirginAmerica, it's, really, aggressive, to, blast, obnoxious, "", entertainment, "", in, your, guests, ', faces, &, amp, ;, they, have, little, recourse]","[@, VirginAmerica, it, 's, really, aggressive, to, blast, obnoxious, ``, entertainment, '', in, your, guests, ', faces, &, amp, ;, they, have, little, recourse]","[@VirginAmerica, it, 's, really, aggressive, to, blast, obnoxious, "", entertainment, "", in, your, guests, ', faces, &, amp, ;, they, have, little, recourse]"
4,@VirginAmerica and it's a really big bad thing about it,"[@VirginAmerica, and, it's, a, really, big, bad, thing, about, it]","[@, VirginAmerica, and, it, 's, a, really, big, bad, thing, about, it]","[@VirginAmerica, and, it, 's, a, really, big, bad, thing, about, it]"


## 2. Where the tokenizers disagree

In [3]:
show_differences(texts)


TWEET: @VirginAmerica What @dhepburn said.
  Regex ( 5): ['@VirginAmerica', 'What', '@dhepburn', 'said', '.']
  NLTK  ( 7): ['@', 'VirginAmerica', 'What', '@', 'dhepburn', 'said', '.']
  spaCy ( 5): ['@VirginAmerica', 'What', '@dhepburn', 'said', '.']

TWEET: @VirginAmerica plus you've added commercials to the experience... tacky.
  Regex (13): ['@VirginAmerica', 'plus', "you've", 'added', 'commercials', 'to', 'the', 'experience', '.', '.', '.', 'tacky', '.']
  NLTK  (13): ['@', 'VirginAmerica', 'plus', 'you', "'ve", 'added', 'commercials', 'to', 'the', 'experience', '...', 'tacky', '.']
  spaCy (12): ['@VirginAmerica', 'plus', 'you', "'ve", 'added', 'commercials', 'to', 'the', 'experience', '...', 'tacky', '.']

TWEET: @VirginAmerica I didn't today... Must mean I need to take another trip!
  Regex (16): ['@VirginAmerica', 'I', "didn't", 'today', '.', '.', '.', 'Must', 'mean', 'I', 'need', 'to', 'take', 'another', 'trip', '!']
  NLTK  (16): ['@', 'VirginAmerica', 'I', 'did', "n't", 't

## 3. Average token count (1000 tweet sample)

In [4]:
s = df['text'].sample(min(1000, len(df)), random_state=42)
pd.Series({n: s.map(fn).map(len).mean() for n, fn in TOKENIZERS.items()}).round(2)

Regex    20.85
NLTK     22.25
spaCy    21.24
dtype: float64

## 4. Tokenization challenges
1. **@mentions and #hashtags** – NLTK splits `@united` into `@` + `united`. Regex keeps it whole. spaCy keeps `@united` but splits `#fail` into `#` + `fail`.
2. **URLs** – NLTK breaks `http://t.co/abc123` into 3 pieces. Regex and spaCy keep it whole.
3. **Emoticons** – `:(` and `:)` stay whole in Regex and spaCy. NLTK splits them into `:` and `)`.
4. **Contractions** – `can't` stays whole in Regex. NLTK and spaCy split it into `ca` + `n't`.
5. **Repeated punctuation** – `!!!` becomes 3 separate tokens in all three tools.
6. **Hyphens** – `co-worker` stays whole in Regex and NLTK. spaCy splits it into 3 tokens.

## 5. Differences between the tokenizers
| | Regex | NLTK | spaCy |
|---|---|---|---|
| Type | Pattern we write | Rule-based (Treebank) | Rule-based with exceptions |
| @mentions | Kept whole | Split | Kept whole |
| #hashtags | Kept whole | Split | Split |
| URLs | Kept whole | Broken | Kept whole |
| Emoticons | Kept whole | Split | Kept whole |
| Contractions | Kept whole | Split | Split |
| Control | Full | Low | Medium |

**Conclusion:** Tweets contain @mentions, hashtags, URLs and emoticons. NLTK is built for clean text, so it breaks them. The custom Regex tokenizer handles tweets best. spaCy is a good second choice.